# 12. L2any L2both diffloop

Part of the **[Fig. 1 chapter](fig1.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
import os
import cooler
import pathlib
import numpy as np
import pandas as pd
from scipy.sparse import load_npz, save_npz, vstack, csr_matrix, triu
from scipy.stats import f, zscore, ranksums
from schicluster.cool.utilities import get_chrom_offsets
from multiprocessing import Pool
from concurrent.futures import ProcessPoolExecutor, as_completed
import warnings
warnings.filterwarnings("ignore")
from glob import glob
from gliderport.preset import notebook_snakemake


In [ ]:
indir = '/data/ENTEx/'
outdir = f'{ENTEX_ROOT}/analysis/'


In [ ]:
meta = pd.read_csv(f'{indir}clustering/merged/group_meta.tsv', header=0, index_col=0, sep='\t')
meta


In [ ]:
# dmr_group = {}
# for l1,l1_df in meta.groupby('L1_annot'):
#     count = len(l1_df['L2_both'].unique())
#     if count>1:
#         dmr_group[l1] = {}
#         for l2b,l2b_df in l1_df.groupby('L2_both'):
#             tmp = list(l2b_df['L2_any'].unique())
#             if len(tmp)>1:
#                 dmr_group[l1][l2b.replace('-c','-b')] = tmp


In [ ]:
dmr_group = {}
for l1,l1_df in meta.groupby('L1_annot'):
    count = len(l1_df['L2_both'].unique())
    if count>1:
        for l2b,l2b_df in l1_df.groupby('L2_both'):
            tmp = list(l2b_df['L2_any'].unique())
            if len(tmp)>1:
                dmr_group[l2b.replace('-c','-b')] = tmp


In [ ]:
print(len(dmr_group))

In [ ]:
group_list = list(dmr_group.keys())
print(len(group_list))


In [ ]:
notebook_snakemake(
    work_dir=f'{outdir}L2any_L2both_diffloop/',
    notebook_dir=f'{outdir}L2any_L2both_diffloop/template/',
    groups=group_list,
    default_cpu=3,
    default_mem_gb=12,
    redo_prepare=True,
)

In [ ]:
!snakemake --snakefile Snakefile -j --keep-going

In [ ]:
group_name = 'c10-b0'


In [ ]:
group_list = dmr_group[group_name]
ctgroup = [[xx] for xx in group_list]
ctgroup

In [ ]:
count = meta.groupby('L2_any')['count'].sum()
ncell = count.loc[group_list].sum()
print(ncell)


In [ ]:
coolpath = pd.read_csv(f'{outdir}coollist_L2any.txt', index_col=None, header=None)[0]
coolpath = pd.Series({xx.split('/')[-1].split('.')[0]:'/'.join(xx.split('/')[:-1]) for xx in coolpath.values})
coolpath

In [ ]:
res = 10000
chrom_size_path = '/ref/hg38/fasta/hg38.main.chrom.sizes'
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes.iloc[:22]
bins_df = cooler.binnify(chrom_sizes, res)
chrom_offset = get_chrom_offsets(bins_df)


In [ ]:
bkl = pd.read_csv('/ref/blacklist/hg38_bismark_loop_blacklist.bed', sep='\t', header=None, index_col=None)

In [ ]:
def compute_anova(c, matrix):
    # c, matrix = args
    ngene = int(chrom_sizes.loc[c] // res) + 1
    bkl_tmp = bkl.loc[(bkl[0]==c), [1,2]].values // res
    cov = np.zeros(ngene)
    for xx,yy in bkl_tmp:
        cov[xx-7:yy+7] = 1
    tot, last = 0, 0
    Esum, E2sum, Elast, E2last, ss_intra = [csr_matrix((ngene, ngene)) for i in range(5)]
    for ctlist in ctgroup:
        for ct in ctlist:
            cool_e = cooler.Cooler(f'{coolpath[ct]}/{ct}.{matrix}.cool')
            E = triu(cool_e.matrix(balance=False, sparse=True).fetch(c))
            cool_e2 = cooler.Cooler(f'{coolpath[ct]}/{ct}.{matrix}2.cool')
            E2 = triu(cool_e2.matrix(balance=False, sparse=True).fetch(c))
            n = cool_e.info['group_n_cells']
            Esum += E * n
            E2sum += E2 * n
            tot += n
            # print(c, ct)
        Egroup = Esum - Elast
        E2group = E2sum - E2last
        Egroup.data = Egroup.data ** 2 / (tot - last)
        ss_intra += (E2group - Egroup)
        Elast = Esum.copy()
        E2last = E2sum.copy()
        last = tot
    Esum.data = Esum.data ** 2 / tot
    ss_total = E2sum - Esum
    ss_intra.data = 1 / ss_intra.data
    ss_total = ss_total.multiply(ss_intra)
    # print(c, ss_total.data.min(), ss_intra.data.min())

    ss_total.data = (ss_total.data - 1) * (tot - len(ctgroup)) / (len(ctgroup) - 1)
    ss_total = ss_total.tocoo()
    bklfilter = np.logical_and(cov[ss_total.row]==0, cov[ss_total.col]==0)
    distfilter = np.logical_and((ss_total.col-ss_total.row)>5, (ss_total.col-ss_total.row)<500)
    idxfilter = np.logical_and(bklfilter, distfilter)
    # print(idxfilter.sum(), len(idxfilter))
    ss_total = csr_matrix((ss_total.data[idxfilter], (ss_total.row[idxfilter], ss_total.col[idxfilter])), (ngene, ngene))
    save_npz(f'{outdir}L2any_L2both_diffloop/{group_name}/{matrix}pv_{c}.npz', ss_total)

    return [c, matrix, tot]



In [ ]:
cpu = 3
with ProcessPoolExecutor(cpu) as executor:
    futures = []
    for x in chrom_sizes.index:
        for y in ['Q', 'E', 'T']:
            future = executor.submit(
                compute_anova,
                c=x,
                matrix=y,
            )
            futures.append(future)

    # result = []
    for future in as_completed(futures):
        # result.append(future.result())
        # c1, c2 = result[-1][0], result[-1][1]
        tmp = future.result()
        print(f'{tmp[0]} {tmp[1]} finished')
        

In [ ]:
def chrom_iterator(input_dir, chrom_order, chrom_offset):
    for chrom in chrom_order:
        output_path = f'{input_dir}_{chrom}.npz'
        if not pathlib.Path(output_path).exists():
            continue
        chunk_size = 5000000
        data = load_npz(output_path).tocoo()
        df = pd.DataFrame({'bin1_id': data.row, 'bin2_id': data.col, 'count': data.data})
        df = df[df['bin1_id'] <= df['bin2_id']]
        for i, chunk_start in enumerate(range(0, df.shape[0], chunk_size)):
            chunk = df.iloc[chunk_start:chunk_start + chunk_size]
            chunk.iloc[:, :2] += chrom_offset[chrom]
            yield chunk


In [ ]:
for matrix in ['Q', 'E', 'T']:
    output_path = f'{outdir}L2any_L2both_diffloop/{group_name}/{matrix}pv'
    cooler.create_cooler(cool_uri=f'{output_path}.cool',
                         bins=bins_df,
                         pixels=chrom_iterator(input_dir=output_path,
                                               chrom_order=chrom_sizes.index,
                                               chrom_offset=chrom_offset
                                              ),
                         ordered=True,
                         dtypes={'count': np.float32})


In [ ]:
os.system(f'rm {outdir}L2any_L2both_diffloop/{group_name}/*pv_c*.npz')


In [ ]:
leg = pd.Index(np.concatenate(ctgroup))
leg

In [ ]:
loopall = [pd.read_csv(f'{coolpath[ct]}/{ct}.loop.bedpe', sep='\t', index_col=None, header=None) for ct in leg]
loopall = pd.concat(loopall, axis=0)
loopall = loopall.drop([6], axis=1).drop_duplicates(subset=[0,1,4]).sort_values([0,1,4])
loopall = pd.concat([loopall[(loopall[0]==c).values] for c in chrom_sizes.index])
loopall.index = np.arange(loopall.shape[0])
loopall


In [ ]:
loopall.to_csv(f'{outdir}L2any_L2both_diffloop/{group_name}/merged_loop.bedpe', sep='\t', index=False, header=False)
loopall.to_hdf(f'{outdir}L2any_L2both_diffloop/{group_name}/merged_loop.hdf', key='data')


In [ ]:
for c in chrom_sizes.index:
    loopfilter = (loopall[0]==c)
    looptmp = loopall.loc[loopfilter, [1,4]].values // res
    for matrix in ['Q', 'E', 'T']:
        cool = cooler.Cooler(f'{outdir}L2any_L2both_diffloop/{group_name}/{matrix}pv.cool')
        pv = triu(cool.matrix(balance=False, sparse=True).fetch(c)).tocsr()
        loopall.loc[loopfilter, f'{matrix}anova'] = pv[(looptmp[:,0], looptmp[:,1])].A1
    print(c)


In [ ]:
from scipy.stats import f
from statsmodels.sandbox.stats.multicomp import multipletests as FDR
    
def stats2fdr(data):
    pv = [f.sf(x, len(leg)-1, ncell-len(leg)) for x in data.values]
    fdr = FDR(pv, 0.01, "fdr_bh")[1]
    return fdr


In [ ]:
cpu = 3
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for m in ['Q', 'E', 'T']:
        future = executor.submit(
            stats2fdr,
            data=loopall[f'{m}anova'],
        )
        futures[future] = m

    for future in as_completed(futures):
        m = futures[future]
        tmp = future.result()
        loopall[f'{m}fdr'] = tmp.copy()
    

In [ ]:
loopall.to_hdf(f'{outdir}L2any_L2both_diffloop/{group_name}/merged_loop.hdf', key='data')


In [ ]:
# get imputed q value at each contact from cool file 
def load_Q(ct, m):
    tmp = []
    cool_file = cooler.Cooler(f'{coolpath[ct]}/{ct}.{matrix}.cool').matrix(balance=False, sparse=True)
    for c in chrom_sizes.index:
        mat = cool_file.fetch(c).tocsr()
        # select for loopall contact q value (as 1d array)
        sel = (loopall[0]==c)
        if sel.sum()>0:
            tmp.append(mat[(loopall.loc[sel, 1].values // res, loopall.loc[sel, 4].values // res)].A1)
        # print(ct, c)
    return np.concatenate(tmp)


In [ ]:
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for xx in leg:
        future = executor.submit(
            load_Q,
            ct=xx,
            m='Q'
        )
        futures[future] = xx

    loopq = []
    for future in as_completed(futures):
        tmp = future.result()
        ct = futures[future]
        loopq.append(pd.DataFrame(tmp, columns=[ct]))
        print(f'{ct} finished')
        

In [ ]:
loopq = pd.concat(loopq, axis=1)
loopq = loopq[leg]


In [ ]:
loopq.to_hdf(f'{outdir}L2any_L2both_diffloop/{group_name}/loop_Q.hdf', key='data')
